Report 5


Anh Do

020416-2317

anhd@kth.se

In [186]:
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
import random
import itertools
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

## Code

In [187]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 2)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [ ]:
# ==========================================================
# 1. OA Model
# ==========================================================

def create_nlp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # FIXED: integer variables must have bounds (0, 3) not (-2, 2)!
    def nlp_bounds_rule(m, i):
        if i >= 5: return (0, 3)  # Continuous relaxation of integers
        return (-2, 2)
    
    m.x = pyo.Var(m.I, bounds=nlp_bounds_rule, domain=pyo.Reals)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Constraints
    m.cons = pyo.ConstraintList()
    m.cons.add(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.cons.add(expr=sum(m.x[i]**2 for i in m.I) - 3 <= 0)

    return m

def create_feas_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    m.x = pyo.Var(m.I, bounds=lambda m, i: (-2, 2), domain=pyo.Reals)
    m.u = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.u, sense=pyo.minimize)
    
    # Relaxed Constraints
    m.cons = pyo.ConstraintList()
    m.cons.add(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= m.u)
    m.cons.add(expr=sum(m.x[i]**2 for i in m.I) - 3 <= m.u)
    return m

def create_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.mu = pyo.Var(domain=pyo.Reals)

    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    m.cuts = pyo.ConstraintList()
    m.ubd_cuts = pyo.ConstraintList()
    return m

def create_master_model_corrected():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.mu = pyo.Var(domain=pyo.Reals)

    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)
    
    m.cuts = pyo.ConstraintList()
    m.ubd_cuts = pyo.ConstraintList()
    return m

# ==========================================================
# Helpers
# ==========================================================
def fix_variables(model, y_values):
    for i in range(5, 9):
        model.x[i].fix(y_values[i])

In [189]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.4f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.4f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.4f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -10.0000     | 17.000000   
2     | -9.7500      | 24.062500   
1     | -10.0000     | 17.000000   
2     | -9.7500      | 24.062500   
3     | -8.2578      | 11.113464   
3     | -8.2578      | 11.113464   
4     | -7.7578      | 12.090027   
4     | -7.7578      | 12.090027   
5     | -7.6156      | 10.688251   
5     | -7.6156      | 10.688251   
6     | -7.1375      | 11.594584   
6     | -7.1375      | 11.594584   
7     | -7.0355      | 5.694672    
7     | -7.0355      | 5.694672    
8     | -6.9912      | 7.421451    
8     | -6.9912      | 7.421451    
9     | -6.4194      | 3.967551    
10    | -6.0143      | 4.314428    
9     | -6.4194      | 3.967551    
10    | -6.0143      | 4.314428    
11    | -6.0000      | 6.797126    
12    | -5.9105      | 4.113303    
11    | -6.0000      | 6.797126    
12    | -5.9105      | 4.113303    
13    | -5.8769      | 3.069151    
14    | -5.8348      | 

In [ ]:
def solve_oa(max_iter=100, tol=1e-3):
    """
    Basic Outer Approximation Algorithm (from the paper).
    """
    
    opt_nlp = SolverFactory('gurobi', manage_env=True)
    opt_master = SolverFactory('gurobi', manage_env=True)
    
    # Initialize
    upper_bound = float('inf')
    lower_bound = -float('inf')
    
    print("="*80)
    print("OUTER APPROXIMATION ALGORITHM")
    print("="*80)
    print(f"{'Iter':<5} | {'Lower Bound':<14} | {'Upper Bound':<14} | {'Gap':<10}")
    print("-" * 55)
    
    # Solve initial NLP with relaxed integer bounds
    m_nlp = create_nlp_model()
    res_nlp = opt_nlp.solve(m_nlp, tee=False)
    
    if res_nlp.solver.termination_condition != pyo.TerminationCondition.optimal:
        print("Initial NLP failed to solve!")
        return None
    
    x_nlp = {i: pyo.value(m_nlp.x[i]) for i in m_nlp.I}
    obj_nlp = pyo.value(m_nlp.obj)
    upper_bound = obj_nlp
    
    # Create master problem
    m_master = create_master_model_corrected()
    
    # Add initial linearization cut
    g1_val = x_nlp[1] + x_nlp[3] + x_nlp[5] + x_nlp[7] - 2
    g1_lhs = g1_val + sum(1.0 * (m_master.x[i] - x_nlp[i]) for i in [1, 3, 5, 7])
    m_master.cuts.add(g1_lhs <= 0)
    
    g2_val = sum(x_nlp[i]**2 for i in m_nlp.I) - 3
    g2_lhs = g2_val + sum(2 * x_nlp[i] * (m_master.x[i] - x_nlp[i]) for i in m_nlp.I)
    m_master.cuts.add(g2_lhs <= 0)
    
    obj_lhs = obj_nlp + sum(-1.0 * (m_master.x[i] - x_nlp[i]) for i in m_master.I)
    m_master.cuts.add(obj_lhs <= m_master.mu)
    
    # OA iterations
    for iteration in range(max_iter):
        # Solve master problem
        res_master = opt_master.solve(m_master, tee=False)
        
        if res_master.solver.termination_condition != pyo.TerminationCondition.optimal:
            break
        
        lower_bound = pyo.value(m_master.obj)
        
        # Get proposed integer solution
        y_proposed = {i: pyo.value(m_master.x[i]) for i in range(5, 9)}
        y_int = tuple([int(round(y_proposed[i])) for i in range(5, 9)])
        
        # Solve NLP with fixed integers
        m_nlp = create_nlp_model()
        for i in range(5, 9):
            m_nlp.x[i].fix(y_int[i-5])
        
        res_nlp = opt_nlp.solve(m_nlp, tee=False)
        
        if res_nlp.solver.termination_condition == pyo.TerminationCondition.optimal:
            x_nlp = {i: pyo.value(m_nlp.x[i]) for i in m_nlp.I}
            obj_nlp = pyo.value(m_nlp.obj)
            
            if obj_nlp < upper_bound:
                upper_bound = obj_nlp
            
            # Add linearization cut
            g1_val = x_nlp[1] + x_nlp[3] + x_nlp[5] + x_nlp[7] - 2
            g1_lhs = g1_val + sum(1.0 * (m_master.x[i] - x_nlp[i]) for i in [1, 3, 5, 7])
            m_master.cuts.add(g1_lhs <= 0)
            
            g2_val = sum(x_nlp[i]**2 for i in m_nlp.I) - 3
            g2_lhs = g2_val + sum(2 * x_nlp[i] * (m_master.x[i] - x_nlp[i]) for i in m_nlp.I)
            m_master.cuts.add(g2_lhs <= 0)
            
            obj_lhs = obj_nlp + sum(-1.0 * (m_master.x[i] - x_nlp[i]) for i in m_master.I)
            m_master.cuts.add(obj_lhs <= m_master.mu)
        
        gap = upper_bound - lower_bound
        print(f"{iteration+1:<5} | {lower_bound:<14.6f} | {upper_bound:<14.6f} | {gap:<10.6f}")
        
        if gap <= tol:
            break
    
    print("=" * 80)
    print(f"FINAL RESULT: Optimal Value = {upper_bound:.6f}")
    print("=" * 80)
    return upper_bound


In [192]:
# Run OA algorithm
optimal_value = solve_oa(max_iter=20, tol=1e-3)


Iter  | Status     | Lower Bound  | Upper Bound 
--------------------------------------------------
1     | NLP Opt    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
1     | NLP Opt    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
2     | NLP Inf    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
2     | NLP Inf    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
3     | NLP Inf    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
3     | NLP Inf    | -10.0000     | -3.4641     
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>
